In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import pickle
import ast
from functools import reduce
from tqdm.auto import tqdm
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}


# sys.path.append(os.path.abspath(R"C:\\Users\\Idener\\Sync\\CODE"))
# from aiddtools import fingerprint_tools as fpt
clear_output()

In [ ]:
import json
import glob
import pandas as pd

rows = []

for file in glob.glob("mibig-json-master/data/*.json"):
    with open(file) as f:
        record = json.load(f)

    cluster = record.get("cluster", {})
    compounds = cluster.get("compounds", [])

    compound_names = []
    compound_smiles = []
    database_ids = []
    npa_ids = []

    for compound in compounds:
        compound_names.append(compound.get("compound"))
        compound_smiles.append(compound.get("chem_struct"))

        for database_id in compound.get("database_id", []):
            database_ids.append(database_id)

            if database_id.lower().startswith("npatlas:"):
                npa_ids.append(database_id.split(":", 1)[1])

    rows.append({"mibig_id": cluster.get("mibig_accession"),
        "biosyn_class": ";".join(cluster.get("biosyn_class", [])),
        "organism": cluster.get("organism_name"),
        "ncbi_tax_id": cluster.get("ncbi_tax_id"),
        "status": cluster.get("status"),
        "minimal": cluster.get("minimal"),
        "n_compounds": len(compounds),
        "compound_names": compound_names,
        "compound_smiles": compound_smiles,
        "database_ids": database_ids,
        "npa_ids": sorted(set(npa_ids))})
df_bgc = pd.DataFrame(rows)

df_bgc = df_bgc.explode('compound_smiles').dropna(subset=['compound_smiles'])
PandasTools.AddMoleculeColumnToFrame(df_bgc, 'compound_smiles')
df_bgc = df_bgc.dropna(subset=['ROMol'])
df_bgc['InChIKey'] = df_bgc['ROMol'].apply(Chem.MolToInchiKey)
df_bgc

# # df_bgc.to_csv("mibig_bgc_df.tsv", sep="\t", index=False)
# mibig_npa = df_bgc.explode("npa_ids")
# mibig_npa = mibig_npa[mibig_npa["npa_ids"].notna()]
# mibig_npa = mibig_npa[(mibig_npa["status"] == "active") &
#     (mibig_npa["minimal"] == False)]
# mibig_npa = mibig_npa.rename(columns={"npa_ids": "npa_id"})
# mibig_npa = mibig_npa[["npa_id", "mibig_id", "biosyn_class", "organism", "ncbi_tax_id"]]
# mibig_npa = mibig_npa.drop_duplicates()

# mibig_npa

In [ ]:
if not os.path.exists('annotator.py'):
    sys.path.append(os.path.abspath('..'))

df_npa = pd.read_csv('/home/mattia/geass/data/NPAtlas_download_2024_09.tsv', sep='\t')
df_npa = df_npa.rename(columns={
    'npaid': 'NPA_ID',
    'compound_inchikey': 'InChIKey',
    'compound_smiles': 'ISOSMILES',
})
PandasTools.AddMoleculeColumnToFrame(df_npa, 'ISOSMILES')
df_npa = df_npa.dropna(subset=['ROMol'])

df_merge = df_bgc.merge(df_npa, on='InChIKey', how='inner', suffixes=('_bgc', '_npa'))
df_merge = df_merge.drop_duplicates(subset=['InChIKey']).reset_index(drop=True)
df_merge['ROMol'] = df_merge['ROMol_npa']
df_merge = df_merge[['mibig_id', 'NPA_ID', 'InChIKey', 'ISOSMILES', 'ROMol']]
df_merge


In [ ]:
from scipy.sparse import csr_matrix, load_npz
from scipy.sparse.csgraph import connected_components

def cluster_from_sparse_similarity(base, df, threshold, method_label):
    ids = np.load(f'{base}_ids.npy', allow_pickle=True)
    sim = load_npz(f'{base}.npz').tocsr().astype(np.float32, copy=False)

    present = df['NPA_ID'].isin(ids)
    df_in = df.loc[present].copy().reset_index(drop=True)
    missing_ids = df.loc[~present, 'NPA_ID'].tolist()
    if missing_ids:
        print(f'Skipping {len(missing_ids)} rows missing from {method_label} ids.')
        print('Examples:', missing_ids[:5])

    order = pd.Index(ids).get_indexer(df_in['NPA_ID'].to_numpy())
    sim = sim[order][:, order].tocsr()
    sim.data = np.clip(sim.data, 0.0, 1.0)

    adj = sim.copy()
    adj.data = (adj.data >= threshold).astype(np.int8)
    adj.setdiag(0)
    adj.eliminate_zeros()

    n_components, labels = connected_components(adj, directed=False, return_labels=True)

    cluster_ids = np.empty(len(labels), dtype=object)
    mapping = {}
    next_id = 1
    for idx, label in enumerate(labels):
        if label not in mapping:
            mapping[label] = str(next_id)
            next_id += 1
        cluster_ids[idx] = mapping[label]

    cluster_col = f'cluster_id_{method_label}_t{int(threshold * 100):02d}'
    out = df_in.copy()
    out[cluster_col] = cluster_ids

    print('Method:', method_label)
    print('Threshold:', threshold)
    print('Clusters:', n_components)
    print('Largest cluster size:', pd.Series(cluster_ids).value_counts().max())
    return out

df_map4 = cluster_from_sparse_similarity('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_map4', df_merge, 0.34, 'map4')
df_map4


In [ ]:
df_sgo = cluster_from_sparse_similarity('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_sgo', df_merge, 0.70, 'sgo')
df_sgo


In [ ]:
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

map4_col = 'cluster_id_map4_t34'
sgo_col = 'cluster_id_sgo_t70'
df_eval = df_map4[['mibig_id', 'NPA_ID', map4_col]].merge(
    df_sgo[['NPA_ID', sgo_col]],
    on='NPA_ID',
    how='inner',
)

def score_against_bgc(df, cluster_col):
    labels_true = df['mibig_id'].astype(str)
    labels_pred = df[cluster_col].astype(str)
    contingency = pd.crosstab(labels_true, labels_pred)
    chi2 = chi2_contingency(contingency, correction=False)[0]
    n = contingency.to_numpy().sum()
    r, k = contingency.shape
    den = n * max(1, min(r - 1, k - 1))
    cramers_v = np.sqrt(chi2 / den) if den > 0 else np.nan
    return {
        'ARI': adjusted_rand_score(labels_true, labels_pred),
        'NMI': normalized_mutual_info_score(labels_true, labels_pred),
        'V_measure': v_measure_score(labels_true, labels_pred),
        'Cramers_V': cramers_v,
        'n_rows': len(df),
        'n_bgc': labels_true.nunique(),
        'n_clusters': labels_pred.nunique(),
    }

summary = pd.DataFrame([
    {'method': 'MAP4', **score_against_bgc(df_eval, map4_col)},
    {'method': 'SGO', **score_against_bgc(df_eval, sgo_col)},
]).set_index('method')

print('Shared rows:', len(df_eval))
summary


In [ ]:
df_cluster_sizes = pd.DataFrame({
    'MAP4': df_map4['cluster_id_map4_t34'].value_counts(),
    'SGO': df_sgo['cluster_id_sgo_t70'].value_counts(),
}).fillna(0).astype(int)

display(df_cluster_sizes.sort_values(['MAP4', 'SGO'], ascending=False).head(20))


In [ ]:
from scipy.sparse import load_npz
from scipy.sparse.csgraph import connected_components
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

def sweep_thresholds(base, df, method_label, thresholds):
    ids = np.load(f'{base}_ids.npy', allow_pickle=True)
    sim = load_npz(f'{base}.npz').tocsr().astype(np.float32, copy=False)
    present = df['NPA_ID'].isin(ids)
    df_in = df.loc[present].copy().reset_index(drop=True)
    order = pd.Index(ids).get_indexer(df_in['NPA_ID'].to_numpy())
    sim = sim[order][:, order].tocsr()
    sim.data = np.clip(sim.data, 0.0, 1.0)

    rows = []
    labels_true = df_in['mibig_id'].astype(str)
    for threshold in thresholds:
        adj = sim.copy()
        adj.data = (adj.data >= threshold).astype(np.int8)
        adj.setdiag(0)
        adj.eliminate_zeros()
        _, labels = connected_components(adj, directed=False, return_labels=True)
        labels_pred = pd.Series(labels).astype(str)
        contingency = pd.crosstab(labels_true, labels_pred)
        chi2 = chi2_contingency(contingency, correction=False)[0]
        n = contingency.to_numpy().sum()
        r, k = contingency.shape
        den = n * max(1, min(r - 1, k - 1))
        cramers_v = np.sqrt(chi2 / den) if den > 0 else np.nan

        rows.append({
            'method': method_label,
            'threshold': threshold,
            'ARI': adjusted_rand_score(labels_true, labels_pred),
            'NMI': normalized_mutual_info_score(labels_true, labels_pred),
            'V_measure': v_measure_score(labels_true, labels_pred),
            'Cramers_V': cramers_v,
            'n_clusters': int(pd.Series(labels).nunique()),
            'largest_cluster': int(pd.Series(labels).value_counts().max()),
        })
    return pd.DataFrame(rows)

map4_thresholds = np.round(np.linspace(0.20, 0.90, 15), 2)
sgo_thresholds = np.round(np.linspace(0.20, 0.90, 15), 2)

df_sweep = pd.concat([
    sweep_thresholds('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_map4', df_merge, 'MAP4', map4_thresholds),
    sweep_thresholds('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_sgo', df_merge, 'SGO', sgo_thresholds),
], ignore_index=True)

df_overview = df_sweep.sort_values(['method', 'threshold']).reset_index(drop=True)
print(df_overview.to_string(index=False))
